# A4.2 - Deep Learning: Feedforward Networks

## Contexte

Classification d'images du dataset **SVHN** (Street View House Numbers) :
- Images RGB 32×32×3
- 50 000 images train, 10 000 images test
- 10 classes (chiffres 0 à 9)
- Objectif : prédire le chiffre au centre de l'image

In [1]:
import os
import site
from pathlib import Path
import numpy as np

# Ensure Jupyter kernel can find pip-installed CUDA shared libraries.
cuda_lib_dirs = []
for site_pkg in site.getsitepackages():
    nvidia_dir = Path(site_pkg) / "nvidia"
    if nvidia_dir.exists():
        for lib_dir in nvidia_dir.glob("*/lib"):
            cuda_lib_dirs.append(str(lib_dir))

if cuda_lib_dirs:
    existing = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(cuda_lib_dirs + ([existing] if existing else []))

import tensorflow as tf
from tensorflow import keras

2026-04-01 10:40:19.605896: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Vérification rapide du support GPU TensorFlow
print("TensorFlow:", tf.__version__)
print("GPU détecté(s):", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU détecté(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Chargement des données (adapter les chemins si nécessaire)
path_train = "data/p4_2026_svhn_train.npz"
path_test = "data/p4_2026_svhn_test.npz"

x_train = np.load(path_train)["images"].transpose(3, 2, 0, 1)
y_train = np.load(path_train)["labels"].squeeze()
x_test = np.load(path_test)["images"].transpose(3, 2, 0, 1)
y_test = np.load(path_test)["labels"].squeeze()

# Convert string labels as int, followed by one-hot encoding
labels = sorted(list(set(y_train)))
y_train = keras.utils.to_categorical([labels.index(x) for x in y_train])
y_test = keras.utils.to_categorical([labels.index(x) for x in y_test])

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")


x_train shape: (50000, 3, 32, 32)
y_train shape: (50000, 10)
x_test shape: (10000, 3, 32, 32)
y_test shape: (10000, 10)


---
## Question 1 : Premier réseau linéaire

Construire un réseau avec :
- **Flatten** layer (32×32×3 → 3072)
- **Dense** output layer avec softmax, kernel et bias initialisés à `RandomNormal`
- Loss : categorical cross-entropy
- Métrique : categorical accuracy
- Optimiseur : Adam, learning rate = $10^{-5}$

Juste définir et compiler le modèle, ne pas le fit.

In [ ]:
# Question 1 : Réseau linéaire
initializer = keras.initializers.RandomNormal()

model_lin = keras.Sequential([
    keras.Input(shape=(3, 32, 32)),
    keras.layers.Flatten(),
    keras.layers.Dense(
        units=10,
        activation="softmax",
        kernel_initializer=initializer,
        bias_initializer=initializer,
    ),
])

model_lin.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"],
)

model_lin.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │        30,730 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,730 (120.04 KB)

 Trainable params: 30,730 (120.04 KB)

 Non-trainable params: 0 (0.00 B)

---
## Question 2 : Model fitting

Entraîner le modèle de la Q1 sur les données train :
- 100 époques
- Batch size par défaut (32)

Sauvegarder en `.keras` et uploader sur INGInious.

In [ ]:
# Question 2 : Entraînement + sauvegarde
training_model_lin = model_lin.fit(
    x_train,
    y_train,
    epochs=100,
    verbose=2,
    validation_data=(x_test, y_test),
)

model_lin.save("q2_linear_model.keras")
print("Modèle sauvegardé: q2_linear_model.keras")

Epoch 1/100


2026-03-31 17:54:08.406516: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 153600000 exceeds 10% of free system memory.
2026-03-31 17:54:08.513302: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 153600000 exceeds 10% of free system memory.


1563/1563 - 13s - 9ms/step - categorical_accuracy: 0.1203 - loss: 107.4600 - val_categorical_accuracy: 0.1226 - val_loss: 89.7020
Epoch 2/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.1265 - loss: 78.7114 - val_categorical_accuracy: 0.1246 - val_loss: 75.0506
Epoch 3/100
1563/1563 - 10s - 7ms/step - categorical_accuracy: 0.1284 - loss: 67.4332 - val_categorical_accuracy: 0.1188 - val_loss: 65.4815
Epoch 4/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.1292 - loss: 59.9546 - val_categorical_accuracy: 0.1257 - val_loss: 57.4689
Epoch 5/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.1323 - loss: 54.3698 - val_categorical_accuracy: 0.1358 - val_loss: 52.9023
Epoch 6/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.1357 - loss: 50.0942 - val_categorical_accuracy: 0.1398 - val_loss: 48.2344
Epoch 7/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.1365 - loss: 46.6571 - val_categorical_accuracy: 0.1334 - val_loss: 45.4314
Epoch 8/100
1563/1563 - 

---
## Question 3 : Performance

- Nombre de paramètres entraînables du réseau ?
- Test accuracy du modèle ?

Format : `number_param, test_acc` (au moins 3 décimales pour l'accuracy)

In [22]:
# Question 3 : Paramètres et accuracy
num_params = model_lin.count_params()
test_loss, test_acc = model_lin.evaluate(x_test, y_test, verbose=0)

print(f"{num_params}, {test_acc:.3f}")

NameError: name 'model_lin' is not defined

---
## Question 4 : Réseau non-linéaire

Ajouter une couche cachée **avant** la couche de sortie :
- Dense, 256 unités, activation **tanh**
- Kernel et bias initialisés à `RandomNormal`
- Tout le reste identique à Q1

Juste définir et compiler, ne pas fit.

In [23]:
# Question 4 : Réseau non-linéaire (tanh)
initializer = keras.initializers.RandomNormal()

model_non_lin = keras.Sequential([
    keras.Input(shape=(3, 32, 32)),
    keras.layers.Flatten(),
    keras.layers.Dense(
        units=256,
        activation="tanh",
        kernel_initializer=initializer,
        bias_initializer=initializer,
    ),
    keras.layers.Dense(
        units=10,
        activation="softmax",
        kernel_initializer=initializer,
        bias_initializer=initializer,
    ),
])

model_non_lin.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"],
)

model_non_lin.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       786,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 789,258 (3.01 MB)

 Trainable params: 789,258 (3.01 MB)

 Non-trainable params: 0 (0.00 B)

---
## Question 5 : Model fitting (non-linéaire)

Entraîner le modèle de la Q4 :
- 100 époques, batch size 32

Sauvegarder en `.keras` et uploader.

In [24]:
# Question 5 : Entraînement réseau non-linéaire + sauvegarde
training_model_non_lin = model_non_lin.fit(
    x_train,
    y_train,
    epochs=100,
    verbose=2,
    validation_data=(x_test, y_test),
)

model_non_lin.save("A4_2_q5_non_linear_model.keras")
print("Modèle sauvegardé: A4_2_q5_non_linear_model.keras")

Epoch 1/100


2026-03-31 18:37:40.943464: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 153600000 exceeds 10% of free system memory.


1563/1563 - 15s - 10ms/step - categorical_accuracy: 0.1717 - loss: 2.2830 - val_categorical_accuracy: 0.2054 - val_loss: 2.2398
Epoch 2/100
1563/1563 - 12s - 7ms/step - categorical_accuracy: 0.1928 - loss: 2.2316 - val_categorical_accuracy: 0.2091 - val_loss: 2.2204
Epoch 3/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.2009 - loss: 2.2216 - val_categorical_accuracy: 0.2173 - val_loss: 2.2058
Epoch 4/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.2056 - loss: 2.2076 - val_categorical_accuracy: 0.2264 - val_loss: 2.1924
Epoch 5/100
1563/1563 - 10s - 6ms/step - categorical_accuracy: 0.2135 - loss: 2.1985 - val_categorical_accuracy: 0.2234 - val_loss: 2.1820
Epoch 6/100
1563/1563 - 11s - 7ms/step - categorical_accuracy: 0.2161 - loss: 2.1886 - val_categorical_accuracy: 0.2268 - val_loss: 2.1719
Epoch 7/100
1563/1563 - 12s - 7ms/step - categorical_accuracy: 0.2179 - loss: 2.1812 - val_categorical_accuracy: 0.2331 - val_loss: 2.1650
Epoch 8/100
1563/1563 - 12s - 7ms/step

---
## Question 6 : Performance (non-linéaire)

- Nombre de paramètres entraînables ?
- Test accuracy ?

Format : `number_param, test_acc`

In [25]:
# Question 6
num_params_non_lin = model_non_lin.count_params()
test_loss_non_lin, test_acc_non_lin = model_non_lin.evaluate(x_test, y_test, verbose=0)

print(f"{num_params_non_lin}, {test_acc_non_lin:.3f}")

789258, 0.528


---
## Question 7 : Fonctions d'activation (tanh vs ReLU)

Même architecture que Q4 mais avec **ReLU** au lieu de tanh.

Effectuer **10 runs** pour chaque modèle (tanh et ReLU), 100 époques, batch size 32.

Format : `tanh_mean_acc, relu_mean_acc` (décimal)

In [27]:
# Question 7 : Comparaison tanh vs ReLU (10 runs chacun)
def build_single_hidden_model(activation_name: str):
    init = keras.initializers.RandomNormal()
    model = keras.Sequential([
        keras.Input(shape=(3, 32, 32)),
        keras.layers.Flatten(),
        keras.layers.Dense(
            256,
            activation=activation_name,
            kernel_initializer=init,
            bias_initializer=init,
        ),
        keras.layers.Dense(
            10,
            activation="softmax",
            kernel_initializer=init,
            bias_initializer=init,
        ),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["categorical_accuracy"],
    )
    return model

def mean_test_accuracy_over_runs(activation_name: str, runs: int = 10):
    accs = []
    for _ in range(runs):
        tf.keras.backend.clear_session()
        model_tmp = build_single_hidden_model(activation_name)
        model_tmp.fit(x_train, y_train, epochs=100, batch_size=32, verbose=0)
        _, test_acc_tmp = model_tmp.evaluate(x_test, y_test, verbose=0)
        accs.append(float(test_acc_tmp))
    return float(np.mean(accs))

tanh_mean_acc = mean_test_accuracy_over_runs("tanh", runs=10)
relu_mean_acc = mean_test_accuracy_over_runs("relu", runs=10)

print(f"{tanh_mean_acc:.3f}, {relu_mean_acc:.3f}")

0.516, 0.660


---
## Question 8 : QCM

Sélectionner les affirmations valides :

- [ ] The ReLU activation function is linear.
- [x] It can be observed on these networks with a single hidden layer that the ReLU activation function and the tanh activation function tend to produce very similar test accuracies (averaged over 10 runs).
- [ ] With a sufficiently small learning rate, the categorical accuracy on the test set is guaranteed to increase after each epoch.
- [x] The only introduced non-linearities in these neural networks come from the activation functions.
- [ ] The kernel and bias initializers do not influence the final solution to question 4. The gradient descent always converges towards the same solution as the minimization problem is convex.
- [ ] The learning rate does not influence a lot the learned neural network, it mainly influences the number of epochs until convergence.
- [ ] With a sufficiently small learning rate, the categorical accuracy on the train set is guaranteed to increase after each epoch.

In [ ]:
# Question 8 : Réflexion

# 2
# 4


---
## Question 9 : Impact du nombre d'unités cachées

Réseau ReLU avec **64**, **256** et **1024** unités cachées.

10 runs pour chaque, 100 époques, batch size 32.

Format : `mean_test_acc_64, mean_test_acc_256, mean_test_acc_1024`

In [ ]:
# Question 9 : Comparaison 64 / 256 / 1024 unités (10 runs)
def build_relu_hidden_units_model(hidden_units: int):
    init = keras.initializers.RandomNormal()
    model_tmp = keras.Sequential([
        keras.Input(shape=(3, 32, 32)),
        keras.layers.Flatten(),
        keras.layers.Dense(
            hidden_units,
            activation="relu",
            kernel_initializer=init,
            bias_initializer=init,
        ),
        keras.layers.Dense(
            10,
            activation="softmax",
            kernel_initializer=init,
            bias_initializer=init,
        ),
    ])
    model_tmp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["categorical_accuracy"],
    )
    return model_tmp

def mean_test_acc_for_hidden_units(hidden_units: int, runs: int = 10):
    accs = []
    print(f"--- Début de l'entraînement pour {hidden_units} unités cachées ---")
    for run_idx in range(runs):
        tf.keras.backend.clear_session()
        model_tmp = build_relu_hidden_units_model(hidden_units)
        # verbose=0 pendant le fit pour ne pas saturer l'affichage d'époques
        model_tmp.fit(x_train, y_train, epochs=100, batch_size=32, verbose=0)
        _, test_acc_tmp = model_tmp.evaluate(x_test, y_test, verbose=0)
        accs.append(float(test_acc_tmp))
        print(f"  -> Run {run_idx+1}/{runs} terminé (Test Acc : {test_acc_tmp:.3f})")
    
    mean_acc = float(np.mean(accs))
    print(f"--- Moyenne finale ({hidden_units} unités) : {mean_acc:.3f} ---\n")
    return mean_acc

# L'évaluation pour 64 et 1024 se fait ici
mean_test_acc_64 = mean_test_acc_for_hidden_units(64, runs=10)
mean_test_acc_1024 = mean_test_acc_for_hidden_units(1024, runs=10)

# Q9 allows reusing the 256-unit ReLU mean from Q7 if it exists.
mean_test_acc_256 = float(0.660)

print(f"\nRÉSULTAT FINAL Q9 : {mean_test_acc_64:.3f}, {mean_test_acc_256:.3f}, {mean_test_acc_1024:.3f}")
# RÉSULTAT FINAL Q9 : 0.584, 0.660, 0.677

--- Début de l'entraînement pour 64 unités cachées ---


2026-04-01 10:43:31.123709: I external/local_xla/xla/service/service.cc:163] XLA service 0x736010006300 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-01 10:43:31.123722: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1050 Ti with Max-Q Design, Compute Capability 6.1
2026-04-01 10:43:31.156094: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-01 10:43:31.502382: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92000
I0000 00:00:1775033012.365760    7616 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  -> Run 1/10 terminé (Test Acc : 0.640)
  -> Run 2/10 terminé (Test Acc : 0.626)
  -> Run 3/10 terminé (Test Acc : 0.619)
  -> Run 4/10 terminé (Test Acc : 0.587)
  -> Run 5/10 terminé (Test Acc : 0.609)
  -> Run 6/10 terminé (Test Acc : 0.627)
  -> Run 7/10 terminé (Test Acc : 0.482)
  -> Run 8/10 terminé (Test Acc : 0.539)
  -> Run 9/10 terminé (Test Acc : 0.493)
  -> Run 10/10 terminé (Test Acc : 0.617)
--- Moyenne finale (64 unités) : 0.584 ---

--- Début de l'entraînement pour 1024 unités cachées ---
  -> Run 1/10 terminé (Test Acc : 0.681)
  -> Run 2/10 terminé (Test Acc : 0.694)
  -> Run 3/10 terminé (Test Acc : 0.659)
  -> Run 4/10 terminé (Test Acc : 0.715)
  -> Run 5/10 terminé (Test Acc : 0.677)
  -> Run 6/10 terminé (Test Acc : 0.654)
  -> Run 7/10 terminé (Test Acc : 0.684)
  -> Run 8/10 terminé (Test Acc : 0.676)
  -> Run 9/10 terminé (Test Acc : 0.667)
  -> Run 10/10 terminé (Test Acc : 0.665)
--- Moyenne finale (1024 unités) : 0.677 ---


RÉSULTAT FINAL Q9 : 0.584, 0.6

---
## Question 10 : Ajout de couches cachées

Comparer les configurations suivantes (ReLU, mêmes hyperparamètres) :

| Index | Configuration |
|-------|---------------|
| 0     | (64, 64)      |
| 1     | (64, 256)     |
| 2     | (256, 64)     |
| 3     | (256, 256)    |
| 4     | (1024, 256)   |
| 5     | (256, 1024)   |
| 6     | (64, 64, 64)  |
| 7     | (256, 256, 256)|

Hint : d'abord 3 runs pour filtrer, puis 10 runs pour les meilleures.

Format : `config_index, test_acc, num_params`

In [ ]:
# Question 10 : Exploration des configurations multi-couches
configs = [
    (64, 64),
    (64, 256),
    (256, 64),
    (256, 256),
    (1024, 256),
    (256, 1024),
    (64, 64, 64),
    (256, 256, 256)
]

def build_multi_hidden_model(hidden_sizes):
    init = keras.initializers.RandomNormal()
    model_tmp = keras.Sequential([
        keras.Input(shape=(3, 32, 32)),
        keras.layers.Flatten(),
    ])
    for size in hidden_sizes:
        model_tmp.add(keras.layers.Dense(
            size,
            activation="relu",
            kernel_initializer=init,
            bias_initializer=init,
        ))
    model_tmp.add(keras.layers.Dense(
        10,
        activation="softmax",
        kernel_initializer=init,
        bias_initializer=init,
    ))
    model_tmp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["categorical_accuracy"],
    )
    return model_tmp

def evaluate_config(hidden_sizes, runs=3):
    accs = []
    for _ in range(runs):
        tf.keras.backend.clear_session()
        model_tmp = build_multi_hidden_model(hidden_sizes)
        # verbose=0 pour ne pas inonder la console lors de l'exécution
        model_tmp.fit(x_train, y_train, epochs=100, batch_size=32, verbose=0)
        _, test_acc_tmp = model_tmp.evaluate(x_test, y_test, verbose=0)
        accs.append(float(test_acc_tmp))
    return float(np.mean(accs))

# Étape 1 : 3 runs pour chaque configuration pour déblayer le terrain
print("Évaluation rapide (3 runs) des configurations :")
results_3_runs = []
for i, config in enumerate(configs):
    print(f"Évaluation de la config {i} {config}...")
    acc = evaluate_config(config, runs=3)
    results_3_runs.append(acc)
    print(f" -> Moyenne (3 runs) : {acc:.3f}")

# Trouver la meilleure configuration sur les 3 runs
best_config_idx = int(np.argmax(results_3_runs))
best_config = configs[best_config_idx]
print(f"\nMeilleure configuration après 3 runs : Index {best_config_idx} -> {best_config}")

# Étape 2 : 10 runs pour la meilleure configuration (si tu veux tester les 2 meilleures tu peux adapter ce code)
print(f"\nÉvaluation approfondie (10 runs) pour la Config {best_config_idx} {best_config} :")
final_acc = evaluate_config(best_config, runs=10)

# Calcul du nombre de paramètres pour le format final
best_model = build_multi_hidden_model(best_config)
num_params = best_model.count_params()

print("\n--- RÉSULTAT FINAL Q10 ---")
print(f"{best_config_idx}, {final_acc:.3f}, {num_params}")


# Datas loaded
# x_train shape: (50000, 3, 32, 32)
# y_train shape: (50000,)
# x_test shape: (10000, 3, 32, 32)
# y_test shape: (10000,)
# Évaluation config 0 (64, 64)...

# I0000 00:00:1775036979.011449      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
# I0000 00:00:1775036979.017174      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
# WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
# I0000 00:00:1775036981.707454     155 service.cc:152] XLA service 0x7b720c004470 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
# I0000 00:00:1775036981.707485     155 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
# I0000 00:00:1775036981.707488     155 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
# I0000 00:00:1775036981.971279     155 cuda_dnn.cc:529] Loaded cuDNN version 91002
# I0000 00:00:1775036982.941792     155 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.

#  -> Moyenne (3 runs) : 0.712
# Évaluation config 1 (64, 256)...
#  -> Moyenne (3 runs) : 0.699
# Évaluation config 2 (256, 64)...
#  -> Moyenne (3 runs) : 0.732
# Évaluation config 3 (256, 256)...
#  -> Moyenne (3 runs) : 0.711
# Évaluation config 4 (1024, 256)...
#  -> Moyenne (3 runs) : 0.718
# Évaluation config 5 (256, 1024)...
#  -> Moyenne (3 runs) : 0.706
# Évaluation config 6 (64, 64, 64)...
#  -> Moyenne (3 runs) : 0.742
# Évaluation config 7 (256, 256, 256)...
#  -> Moyenne (3 runs) : 0.750

# Évaluation 10 runs pour Config 7...

# --- RÉSULTAT FINAL Q10 ---
# 7, 0.748, 920842



---
## Question 11 : QCM (couches cachées)

Sélectionner les affirmations valides :

- [ ] The number of units of the first hidden layer influences the most the time taken by each training epoch because the input layer has a large number of units.
- [ ] The network with 2 hidden layers of 64 units performs better than the linear model of question 2.
- [ ] Increasing the number of trainable parameters (by adding layers and/or number of hidden units) always leads to improve the test accuracy on this dataset.
- [ ] The test accuracy of the network with a single hidden layer and 1024 units cannot be improved by adding other hidden layers.
- [ ] Neural networks with 2 hidden layers are universal approximators, which means that the test accuracy will converge to 100% given enough hidden units. We observe this phenomenon here as increasing the number of hidden units tend to increase the test accuracy.
- [ ] Considering deeper networks, by increasing the number of hidden layers (up to 3), tends to improve the test accuracy on this dataset.

In [ ]:
# Question 11 : Réponses au QCM
# VRAI : The number of units of the first hidden layer influences the most the time taken by each training epoch because the input layer has a large number of units.
# (La matrice Flatten est énorme : 32x32x3 = 3072 features. La majorité du calcul est donc le produit matriciel de l'input et la première couche.)

# VRAI : The network with 2 hidden layers of 64 units performs better than the linear model of question 2.
# (Le modèle linéaire de la Q2 est très primitif, l'ajout de couches de ReLU aide grandement à la séparation non-linéaire sur les images.)

# VRAI : Considering deeper networks, by increasing the number of hidden layers (up to 3), tends to improve the test accuracy on this dataset.
# (En général on voit qu'un réseau plus profond à capacités paramétriques équivalentes tire de meilleures abstractions visuelles qu'une seule couche plate.)

# FAUX : Increasing the number of trainable parameters always leads to improve the test accuracy (sur-apprentissage / overfitting possible).
# FAUX : ... cannot be improved by adding other hidden layers.
# FAUX : test accuracy will converge to 100% (c'est l'accuracy en "training" qui s'approche de 100% avec l'approximation de données par coeur, mais la "test" plafonnera).
